In [ ]:
import os

# Controlla se Conda è già presente nella sessione corrente
if not os.path.exists("/usr/local/miniconda"):
    print("⚙️ Inizializzazione ambiente Conda e mmdet3d...")
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    !bash /tmp/miniconda.sh -b -p /usr/local/miniconda
    os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
    
    !conda create -n openmmlab python=3.10 -y
    !conda run -n openmmlab pip install "numpy<2"
    !conda run -n openmmlab pip install torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118
    !conda run -n openmmlab pip install -U openmim
    !conda run -n openmmlab mim install mmengine "mmcv>=2.0.0rc4,<2.1.0" "mmdet>=3.0.0,<3.1.0"
    
    if not os.path.exists('/content/mmdetection3d'):
        !git clone https://github.com/open-mmlab/mmdetection3d.git -b dev-1.x /content/mmdetection3d
        %cd /content/mmdetection3d
        !conda run -n openmmlab pip install --no-build-isolation .
        %cd /content
    print("✅ Ambiente creato con successo!")
else:
    os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]
    print("⚡ Ambiente Conda già pronto!")

⚡ Ambiente Conda già pronto!


In [2]:
!conda run -n openmmlab python -c "import torch, mmcv, mmengine, mmdet, mmdet3d; print('torch:', torch.__version__); print('mmcv:', mmcv.__version__); print('mmengine:', mmengine.__version__); print('mmdet:', mmdet.__version__); print('mmdet3d:', mmdet3d.__version__); print('CUDA:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')"

torch: 2.0.1+cu118
mmcv: 2.0.1
mmengine: 0.10.7
mmdet: 3.0.0
mmdet3d: 1.4.0
CUDA: False
GPU: NONE


In [4]:
!conda run -n openmmlab pip install ipykernel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 55.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.2/888.2 kB 35.3 MB/s  0:00:00



In [6]:
!conda run -n openmmlab python -m ipykernel install \
    --user \
    --name openmmlab \
    --display-name "Python (OpenMMLab)"

Installed kernelspec openmmlab in /root/.local/share/jupyter/kernels/openmmlab


In [1]:
import sys
import os

# 1. Rileva se l'esecuzione avviene su Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("⚡ Esecuzione su Google Colab.")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # URL del tuo repository GitHub e percorso in Colab
    REPO_URL = "https://github.com/giacomosalvatori03/3D_Perception.git"
    REPO_DIR = "/content/3D_Perception"
    
    # Se la repo non è ancora presente su Colab la clona, altrimenti scarica gli aggiornamenti
    if not os.path.exists(REPO_DIR):
        print("📥 Clonazione del repository da GitHub...")
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print("🔄 Aggiornamento del codice da GitHub...")
        %cd {REPO_DIR}
        !git pull
        
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # PERCORSO DEL DATASET SU GOOGLE DRIVE
    # Modifica questo percorso con la posizione esatta della cartella 'kitti_validation' sul tuo Drive
    DATASET_PATH = "/content/drive/MyDrive/3D_Perception/data/kitti_validation"

else:
    print("💻 Esecuzione in Locale.")
    PROJECT_ROOT = os.path.abspath('..')
    if PROJECT_ROOT not in sys.path:
        sys.path.append(PROJECT_ROOT)
    DATASET_PATH = os.path.join(PROJECT_ROOT, 'data/kitti_validation')

print(f"✅ Directory corrente: {os.getcwd()}")
print(f"✅ Dataset Path: {DATASET_PATH}")

⚡ Esecuzione su Google Colab.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔄 Aggiornamento del codice da GitHub...
/content/3D_Perception
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 8 (delta 2), reused 8 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 698.17 KiB | 3.77 MiB/s, done.
From https://github.com/giacomosalvatori03/3D_Perception
   1a5b821..8efb53c  main       -> origin/main
Updating 1a5b821..8efb53c
Fast-forward
 notebooks/prova.ipynb           |   54 +-
 notebooks/prova2.ipynb          | 1029 +++++++++++++++++++++++++++++++++++++++
 src/detectors/lidar_detector.py |  229 +++------
 3 files changed, 1139 insertions(+), 173 deletions(-)
 create mode 100644 notebooks/prova2.ipynb
✅ Directory corrente: /content/3D_Perception
✅ Dataset Path: /content/drive/MyDrive/3D_Perception/da

In [12]:
%%bash
source /usr/local/miniconda/bin/activate openmmlab

python -c "
from src import KittiDataset, LidarDetector

DATASET_PATH = '/content/drive/MyDrive/3D_Perception/data/kitti_validation'

dataset = KittiDataset(data_root=DATASET_PATH)
detector = LidarDetector(conf_threshold=0.3)

sample = dataset[0]
detections = detector.detect(sample)

print(f'🎉 Inferenza completata con mmdet3d! Oggetti 3D rilevati: {len(detections)}')
for d in detections:
    print('  •', d)
"


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<string>", line 2, in <module>
  File "/content/3D_Perception/src/__init__.py", line 5, in <module>
    from src.detectors import BaseDetector, LidarDetector
  File "/content/3D_Perception/src/detectors/__init__.py", line 2, in <module>
    from .lidar_detector import LidarDetector
  File "/content/3D_Perception/src/detectors/lidar_detector.py", line 6, in <module>
    from mmdet3d.apis import init_model, inference_detector
  File "/usr/local/miniconda/envs/openmmlab/lib/python3.10/site-packages/mmdet3d/apis/__init

CalledProcessError: Command 'b'source /usr/local/miniconda/bin/activate openmmlab\n\npython -c "\nfrom src import KittiDataset, LidarDetector\n\nDATASET_PATH = \'/content/drive/MyDrive/3D_Perception/data/kitti_validation\'\n\ndataset = KittiDataset(data_root=DATASET_PATH)\ndetector = LidarDetector(conf_threshold=0.3)\n\nsample = dataset[0]\ndetections = detector.detect(sample)\n\nprint(f\'\xf0\x9f\x8e\x89 Inferenza completata con mmdet3d! Oggetti 3D rilevati: {len(detections)}\')\nfor d in detections:\n    print(\'  \xe2\x80\xa2\', d)\n"\n'' returned non-zero exit status 1.